# M10 L10: Evaluation and RAGAS-style Metrics

**Build a tiny RAGAS-style eval harness: 3 metrics, golden set, regression detector.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

"The model is good" is not a metric. To know if a model is actually good, you need a **golden set** -- a list of questions with the right answers -- and a **rubric** -- what counts as a good answer.

The standard rubric for RAG is **RAGAS** (Retrieval-Augmented Generation Assessment), which checks three things: (1) is the answer **faithful** to the documents (no hallucination), (2) is the answer **relevant** to the question, and (3) did the search find the **right documents**.

## The 5 things to know

1. **Three RAGAS metrics**: faithfulness, answer_relevancy, context_recall.
2. **The golden set is the contract** -- 100-300 hand-labeled examples.
3. **LLM-as-judge** -- use a strong LLM to grade; spot-check 20 examples per month.
4. **Regression detector** -- compare current run to baseline; alert on >5pp drop.
5. **Stratify by category** -- per-category scores catch what mean masks.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm10-evaluation' / 'L10-evaluation-ragas.py'
_spec = importlib.util.spec_from_file_location('L10', _p)
_L10 = importlib.util.module_from_spec(_spec)
sys.modules['L10'] = _L10
_spec.loader.exec_module(_L10)

print(f'Setup OK. Lesson module: {_L10.LESSON_NUMBER}')

Setup OK. Lesson module: M10.L10


In [2]:
# Run the 15 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L10, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 28 tests...

  PASS  test_detect_regressions_alert
  PASS  test_detect_regressions_handles_equal_metrics
  PASS  test_detect_regressions_no_regression
  PASS  test_detect_regressions_returns_correct_count
  PASS  test_detect_regressions_strict_threshold
  PASS  test_eval_math_invariants
  PASS  test_eval_result_fail
  PASS  test_eval_result_pass
  PASS  test_evaluate_latency_ms_is_deterministic_per_seed
  PASS  test_evaluate_latency_ms_returns_stats
  PASS  test_evaluate_latency_ms_within_budget
  PASS  test_golden_set_by_category
  PASS  test_golden_set_default_len_is_zero
  PASS  test_golden_set_len
  PASS  test_judge_answer_relevancy_match
  PASS  test_judge_answer_relevancy_mismatch
  PASS  test_judge_context_recall_empty_relevant
  PASS  test_judge_context_recall_perfect
  PASS  test_judge_context_recall_zero
  PASS  test_judge_faithfulness_supported
  PASS  test_judge_faithfulness_unsupported
  PASS  test_pass_rate_all_pass
  PASS  test_pass_rate_none_pass
  PASS  test_ra

In [3]:
# === Section 1: golden set ===
print(f'Golden set size: {len(_L10.GOLDEN_SET)}')
cats = _L10.GOLDEN_SET.by_category()
for cat, exs in cats.items():
    print(f'  {cat:<14} {len(exs)} examples')

Golden set size: 7
  factual        5 examples
  procedural     2 examples


In [4]:
# === Section 2: run evaluation ===
judge = _L10.MockJudge()
report = _L10.evaluate_rag(judge, _L10.GOLDEN_SET)
print(f'Evaluated: {len(report.results)} examples')

Evaluated: 7 examples


In [5]:
# === Section 3: per-example scores ===
print(f'\n{"query":<42}  {"faith":>6}  {"rel":>6}  {"recall":>6}  pass?')
for r in report.results:
    q = r.query[:40]
    ok = 'OK' if r.passed else 'MISS'
    print(f'  {q:<42}  {r.scores.faithfulness:>6.2f}  {r.scores.answer_relevancy:>6.2f}  {r.scores.context_recall:>6.2f}  {ok}')


query                                        faith     rel  recall  pass?
  how much vacation do I get per year           1.00    0.38    1.00  MISS
  what is the 401k match                        1.00    0.80    1.00  OK
  sick leave doctor's note rule                 1.00    0.83    1.00  OK
  how long do I have to submit expenses         1.00    0.12    0.00  MISS
  code review SLA                               1.00    0.33    1.00  MISS
  P1 incident response time                     1.00    0.25    1.00  MISS
  data retention policy                         1.00    0.33    0.00  MISS


In [6]:
# === Section 4: aggregate scores ===
m = report.mean_scores()
s = report.std_scores()
print(f'\n{"metric":<20}  {"mean":>6}  {"std":>6}')
print(f'  {"faithfulness":<20}  {m.faithfulness:>6.2f}  {s.faithfulness:>6.2f}')
print(f'  {"answer_relevancy":<20}  {m.answer_relevancy:>6.2f}  {s.answer_relevancy:>6.2f}')
print(f'  {"context_recall":<20}  {m.context_recall:>6.2f}  {s.context_recall:>6.2f}')
print(f'\n  pass rate: {report.pass_rate():.0%}')


metric                  mean     std
  faithfulness            1.00    0.00
  answer_relevancy        0.44    0.27
  context_recall          0.71    0.49

  pass rate: 29%


In [7]:
# === Section 5: by category ===
print('\nBy category:')
for cat, scores in report.by_category().items():
    print(f'  {cat:<14}  faith={scores.faithfulness:.2f}  rel={scores.answer_relevancy:.2f}  recall={scores.context_recall:.2f}')


By category:
  factual         faith=1.00  rel=0.49  recall=0.60
  procedural      faith=1.00  rel=0.29  recall=1.00


In [8]:
# === Section 6: regression detector ===
fake_baseline = _L10.RAGASScores(faithfulness=0.90, answer_relevancy=0.85, context_recall=0.95)
alerts = _L10.detect_regressions(fake_baseline, m, threshold=0.05)
print('\nRegression check (vs simulated baseline):\n')
for a in alerts:
    print(f'  {a}')


Regression check (vs simulated baseline):

  [OK] faithfulness: baseline=0.900  current=1.000  delta=+0.100
  [REGRESSION] answer_relevancy: baseline=0.850  current=0.436  delta=-0.414
  [REGRESSION] context_recall: baseline=0.950  current=0.714  delta=-0.236


In [9]:
# === Section 7: judge internals ===
print('\nMock judge internals:')
print(f'  faithfulness("the cat is on the mat", "the cat is on the mat") = {judge.judge_faithfulness("the cat is on the mat", "the cat is on the mat"):.3f}')
print(f'  faithfulness("quantum entanglement is purple", "the cat is on the mat") = {judge.judge_faithfulness("quantum entanglement is purple", "the cat is on the mat"):.3f}')
print(f'  relevancy("you get 20 days vacation", "how much vacation do I get") = {judge.judge_answer_relevancy("you get 20 days vacation", "how much vacation do I get"):.3f}')
print(f'  context_recall(["d1", "d2"], ["d1"]) = {judge.judge_context_recall(["d1", "d2"], ["d1"]):.3f}')


Mock judge internals:
  faithfulness("the cat is on the mat", "the cat is on the mat") = 1.000
  faithfulness("quantum entanglement is purple", "the cat is on the mat") = 0.375
  relevancy("you get 20 days vacation", "how much vacation do I get") = 0.333
  context_recall(["d1", "d2"], ["d1"]) = 1.000


## Architecture (Mermaid)

```mermaid
flowchart LR
    GS[Golden Set] --> RUN[eval run]
    RAG[Mock RAG / real] --> RUN
    RUN --> J[LLM Judge]
    J --> R[Report]
    R --> RD[Regression detector]
    GS --> RD
    BASE[Baseline] --> RD
    RD --> ALERT[Alert if drop > 5pp]
```

**Read as:** Golden set + RAG + judge = report; compare to baseline; alert on regression.

## Reflect (10 min)

1. What does each of the 3 RAGAS metrics catch that the others miss?
2. Why is the golden set called "the contract"? What happens when it changes?
3. When is a regression NOT a regression? Give 3 reasons.
4. The LLM-as-judge is itself an LLM. What biases might it have?
5. (Stretch) Add a `context_precision` metric (fraction of retrieved that are relevant).

---

**Read next:**
- `lessons/m11-observability/L11-observability-cost.ipynb` -- metrics, traces, alerts
- `lessons/m12-security/L12-security-acl-injection.ipynb` -- prompt injection, ACL
- `SYLLABUS.md` -- the 18 modules and 21-day plan